# IoT Intrusion Detection Project
## Milestone 2 — Data Preprocessing

This notebook corresponds to **Steps 2 and 3 of the brief**:
- *"Build safe train, validation, and test splits."*
- *"Prepare the features — scaling numeric features, checking for leakage, and making sure the
  exact same preprocessing is applied at training time and inference time."*

**Order of operations (leakage-safe):**
1. Load the cleaned dataset (NaN and inf still present).
2. Replace inf with NaN.
3. Split into train / val / test with an ordered per-class 70/15/15 split.
4. Fit the imputer on **train only**, then transform val and test.
5. Fit the scaler on **train only**, then transform val and test.
6. Save the fitted imputer, scaler, label encoder, and feature-name list alongside the
   numpy arrays. These artifacts are consumed by Milestone 3 (training) and Milestone 4
   (the inference endpoint, which must reproduce preprocessing byte-for-byte).

**Safe split choice:** explicit timestamps are not available in the cleaned feature files, so we
preserve the original row order inside each class and take contiguous 70/15/15 chunks. This is
safer than random row-level splitting because adjacent, highly similar flow records are less
likely to appear in both training and testing.


## 1. Setup

In [3]:
import os
import json
import numpy as np
import pandas as pd
import joblib

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer


## 2. Load cleaned dataset

In [4]:
df = pd.read_csv("../data/processed/clean_dataset.csv")
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (1085047, 40)


,Header_Length,Protocol Type,Time_To_Live,Rate,fin_flag_number,syn_flag_number,rst_flag_number,psh_flag_number,ack_flag_number,ece_flag_number,...,Tot sum,Min,Max,AVG,Std,Tot size,IAT,Number,Variance,label
0,32.0,6,198.4,430.273287,0.0,0.0,0.0,0.0,1.0,0.0,...,5004,66,2962,500.4,977.325512,500.4,0.002324,10,9.551652e+05,BenignTraffic
1,22.4,6,114.4,484.616113,0.0,0.0,0.0,0.2,0.6,0.0,...,1404,60,583,140.4,174.827026,140.4,0.002179,10,3.056449e+04,BenignTraffic
2,27.2,6,62.5,398.610950,0.0,0.0,0.0,0.0,0.8,0.0,...,10784,60,1514,1078.4,701.384536,1078.4,0.003173,10,4.919403e+05,BenignTraffic
3,23.2,6,62.1,425.187438,0.0,0.1,0.0,0.0,0.6,0.0,...,7884,60,2962,788.4,1025.156812,788.4,0.002507,10,1.050946e+06,BenignTraffic
4,32.0,6,64.0,1563.521956,0.0,0.0,0.0,0.1,1.0,0.0,...,13782,156,1514,1378.2,429.437306,1378.2,0.000660,10,1.844164e+05,BenignTraffic


## 3. Replace infinities with NaN

This is a *structural* fix (inf is never a valid feature value) and does not depend on any split,
so it's safe to do before splitting. Actual imputation happens after the split.

In [5]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
n_inf = np.isinf(df[numeric_cols]).sum().sum()
print(f"Infinite values before replacement: {n_inf}")

df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)

n_nan = df[numeric_cols].isnull().sum().sum()
print(f"NaN values after replacing inf (to be imputed on train only): {n_nan}")

Infinite values before replacement: 46
NaN values after replacing inf (to be imputed on train only): 130


## 4. Separate features and target

In [6]:
X = df.drop("label", axis=1)
y = df["label"]

feature_names = list(X.columns)
print("Features shape:", X.shape)
print("Target shape:  ", y.shape)
print("Number of features:", len(feature_names))

Features shape: (1085047, 39)
Target shape:   (1085047,)
Number of features: 39


## 5. Encode labels

`LabelEncoder` assigns integer class indices in alphabetical order. We save the fitted encoder
so Milestone 4's inference endpoint can map model output back to human-readable class names.

In [7]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Classes (index -> name):")
for i, name in enumerate(label_encoder.classes_):
    print(f"  {i} -> {name}")

Classes (index -> name):
  0 -> BenignTraffic
  1 -> DDoS-ICMP_Flood
  2 -> DictionaryBruteForce
  3 -> DoS-SYN_Flood
  4 -> MITM-ArpSpoofing
  5 -> Mirai-greeth_flood
  6 -> Recon-PortScan
  7 -> SqlInjection


## 6. Train / Validation / Test split

70 / 15 / 15 split inside each encoded class, preserving original row order instead of
randomizing rows. Since the feature files do not include timestamps, this ordered per-class
split is the practical leakage-reduction strategy: nearby records from the same attack run are
kept together more often than with random stratification.


In [8]:
def ordered_per_class_split(X, y, train_frac=0.70, val_frac=0.15):
    """Return train/val/test splits using contiguous row-order chunks per class."""
    y_arr = np.asarray(y)
    train_idx, val_idx, test_idx = [], [], []

    for class_id in np.unique(y_arr):
        class_idx = np.flatnonzero(y_arr == class_id)
        train_end = int(len(class_idx) * train_frac)
        val_end = int(len(class_idx) * (train_frac + val_frac))

        train_idx.append(class_idx[:train_end])
        val_idx.append(class_idx[train_end:val_end])
        test_idx.append(class_idx[val_end:])

    train_idx = np.concatenate(train_idx)
    val_idx = np.concatenate(val_idx)
    test_idx = np.concatenate(test_idx)

    return (
        X.iloc[train_idx].copy(), y_arr[train_idx],
        X.iloc[val_idx].copy(), y_arr[val_idx],
        X.iloc[test_idx].copy(), y_arr[test_idx],
    )


X_train, y_train, X_val, y_val, X_test, y_test = ordered_per_class_split(X, y_encoded)

print(f"Train:      {X_train.shape}  ({len(y_train):,} rows)")
print(f"Validation: {X_val.shape}  ({len(y_val):,} rows)")
print(f"Test:       {X_test.shape}  ({len(y_test):,} rows)")


Train:      (759529, 39)  (759,529 rows)
Validation: (162758, 39)  (162,758 rows)
Test:       (162760, 39)  (162,760 rows)


## 7. Impute missing values — fit on train only

Mean imputation. The imputer is fit **only on the training split** and then applied to val and
test. This closes the leakage loophole that would exist if the imputation mean were computed
over the full dataset.

In [9]:
imputer = SimpleImputer(strategy="mean")

X_train_imp = imputer.fit_transform(X_train)   # fit on TRAIN only
X_val_imp   = imputer.transform(X_val)
X_test_imp  = imputer.transform(X_test)

# Sanity check: no NaN should remain
assert not np.isnan(X_train_imp).any(), "NaN left in X_train after imputation"
assert not np.isnan(X_val_imp).any(),   "NaN left in X_val after imputation"
assert not np.isnan(X_test_imp).any(),  "NaN left in X_test after imputation"
print("Imputation complete, no NaN remaining in any split.")

Imputation complete, no NaN remaining in any split.


## 8. Scale features — fit on train only

Standard scaling (zero mean, unit variance). Again, fit on training data only.

In [10]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imp)   # fit on TRAIN only
X_val_scaled   = scaler.transform(X_val_imp)
X_test_scaled  = scaler.transform(X_test_imp)

print("X_train_scaled:", X_train_scaled.shape)
print("X_val_scaled:  ", X_val_scaled.shape)
print("X_test_scaled: ", X_test_scaled.shape)

X_train_scaled: (759529, 39)
X_val_scaled:   (162758, 39)
X_test_scaled:  (162760, 39)


## 9. Save arrays and preprocessing artifacts

Three categories of output:

1. **Numpy arrays** for Milestone 3 (model training).
2. **Fitted preprocessing objects** (imputer, scaler, label encoder) for Milestone 4 (inference
   endpoint). Without these, the deployed service cannot reproduce training-time preprocessing.
3. **Feature-name list** for Milestone 4. When the endpoint receives a JSON payload, it needs to
   assemble the columns in the exact order the scaler was fit on.

In [11]:
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

# 1. Arrays for Milestone 3
np.save("../data/processed/X_train_scaled.npy", X_train_scaled)
np.save("../data/processed/X_val_scaled.npy",   X_val_scaled)
np.save("../data/processed/X_test_scaled.npy",  X_test_scaled)
np.save("../data/processed/y_train.npy",        y_train)
np.save("../data/processed/y_val.npy",          y_val)
np.save("../data/processed/y_test.npy",         y_test)

# 2. Fitted preprocessing objects for Milestone 4 inference
joblib.dump(imputer,       "../models/imputer.joblib")
joblib.dump(scaler,        "../models/scaler.joblib")
joblib.dump(label_encoder, "../models/label_encoder.joblib")

# 3. Feature-name list for Milestone 4 inference (column order matters!)
with open("../models/feature_names.json", "w") as f:
    json.dump(feature_names, f, indent=2)

print("Saved arrays to ../data/processed/")
print("Saved preprocessing artifacts to ../models/")
print("  - imputer.joblib")
print("  - scaler.joblib")
print("  - label_encoder.joblib")
print("  - feature_names.json")

Saved arrays to ../data/processed/
Saved preprocessing artifacts to ../models/
  - imputer.joblib
  - scaler.joblib
  - label_encoder.joblib
  - feature_names.json


## Summary

- Replaced infinities with NaN *before* splitting (structural fix, no statistics involved).
- Ordered per-class 70/15/15 split — chosen because explicit timestamps are not available in
  the cleaned feature files. This preserves original row order within each class and reduces
  near-duplicate leakage compared with random row-level splitting.
- Imputer (mean) and scaler (StandardScaler) were **fit on the training split only** and then
  applied unchanged to val and test — no preprocessing leakage.
- All preprocessing artifacts (imputer, scaler, label encoder, feature-name list) are persisted
  to `../models/` so the Milestone 4 inference endpoint can reproduce the exact same
  preprocessing at serve time.
